# 01 — Data Preparation

Cleans the raw Spotify dataset and extracts playlist plays into tabular form. Downstream feature selection (02) and feature engineering (03) consume these two files.

**Inputs** (`data/`):
- `data/Spotify_dataset_gigasheet.csv` — raw Spotify audio-feature catalogue
- `data/playlist.json` — raw Million Playlist Dataset slice (1,000 playlists)

**Outputs** (`outputs/`):
- `outputs/spotify_cleaned.csv` — cleaned catalogue (19,675 songs × 25 cols)
- `outputs/USETHIS_output_filtered.csv` — playlist plays whose tracks appear in the catalogue (15,011 rows)

# 1. Data Cleaning


In [1]:
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
df = pd.read_csv("./data/Spotify_dataset_gigasheet.csv")
df

,Artist,Track,Album,Album_type,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,Title,Channel,Views,Likes,Comments,Licensed,official_video,Stream,EnergyLiveness,most_playedon
0,Gorillaz,Feel Good Inc.,Demon Days,album,0.818,0.705,-6.679,0.1770,0.008360,0.002330,...,Gorillaz - Feel Good Inc. (Official Video),Gorillaz,693555221,6220896,169907,True,True,1040234854,1.150082,Spotify
1,Gorillaz,Rhinestone Eyes,Plastic Beach,album,0.676,0.703,-5.815,0.0302,0.086900,0.000687,...,Gorillaz - Rhinestone Eyes [Storyboard Film] (...,Gorillaz,72011645,1079128,31003,True,True,310083733,15.183585,Spotify
2,Gorillaz,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,0.695,0.923,-3.930,0.0522,0.042500,0.046900,...,Gorillaz - New Gold ft. Tame Impala & Bootie B...,Gorillaz,8435055,282142,7399,True,True,63063467,7.956897,Spotify
3,Gorillaz,On Melancholy Hill,Plastic Beach,album,0.689,0.739,-5.810,0.0260,0.000015,0.509000,...,Gorillaz - On Melancholy Hill (Official Video),Gorillaz,211754952,1788577,55229,True,True,434663559,11.546875,Spotify
4,Gorillaz,Clint Eastwood,Gorillaz,album,0.663,0.694,-8.627,0.1710,0.025300,0.000000,...,Gorillaz - Clint Eastwood (Official Video),Gorillaz,618480958,6197318,155930,True,True,617259738,9.942693,Youtube
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20589,SICK LEGEND,JUST DANCE HARDSTYLE,JUST DANCE HARDSTYLE,single,0.582,0.926,-6.344,0.0328,0.448000,0.000000,...,JUST DANCE HARDSTYLE,SICK LEGEND - Topic,71678,1113,0,True,True,9227144,11.036949,Spotify
20590,SICK LEGEND,SET FIRE TO THE RAIN HARDSTYLE,SET FIRE TO THE RAIN HARDSTYLE,single,0.531,0.936,-1.786,0.1370,0.028000,0.000000,...,SET FIRE TO THE RAIN HARDSTYLE,SICK LEGEND - Topic,164741,2019,0,True,True,10898176,10.140845,Spotify
20591,SICK LEGEND,OUTSIDE HARDSTYLE SPED UP,OUTSIDE HARDSTYLE SPED UP,single,0.443,0.830,-4.679,0.0647,0.024300,0.000000,...,OUTSIDE HARDSTYLE SPED UP,SICK LEGEND - Topic,35646,329,0,True,True,6226110,5.389610,Spotify
20592,SICK LEGEND,ONLY GIRL HARDSTYLE,ONLY GIRL HARDSTYLE,single,0.417,0.767,-4.004,0.4190,0.356000,0.018400,...,ONLY GIRL HARDSTYLE,SICK LEGEND - Topic,6533,88,0,True,True,6873961,7.101852,Spotify


In [3]:
print((df['Title'] == 'None').sum())
df.isna().sum()

0


Artist              0
Track               0
Album               0
Album_type          0
Danceability        0
Energy              0
Loudness            0
Speechiness         0
Acousticness        0
Instrumentalness    0
Liveness            0
Valence             0
Tempo               0
Duration_min        0
Title               0
Channel             0
Views               0
Likes               0
Comments            0
Licensed            0
official_video      0
Stream              0
EnergyLiveness      0
most_playedon       0
dtype: int64

In [4]:
df['Album_type'].value_counts()

Album_type
album          14834
single          4973
compilation      787
Name: count, dtype: int64

In [5]:
#remove compilations
df = df[(df['Album_type'] != "compilation")].copy()

In [6]:
df['Duration_min'].sort_values()

13773     0.000000
11824     0.000000
11860     0.516417
20521     0.516667
20530     0.516667
           ...    
5814     16.390533
5359     23.799650
5361     24.737667
4427     55.677867
9311     68.670967
Name: Duration_min, Length: 19807, dtype: float64

In [7]:
# remove tracks longer than 10 mins
df = df[(df['Duration_min'] < 10)].copy()

In [8]:
df = df.sort_values('Stream', ascending=False)
df.duplicated(subset=['Artist', 'Track']).value_counts()

False    19693
True        55
Name: count, dtype: int64

In [9]:
# remove duplicate artist/track combos
df = df.drop_duplicates(subset=['Artist', 'Track'], keep='first')

In [10]:
numerical_cols = df.select_dtypes(include='number').columns
zero_counts = (df[numerical_cols] == 0).sum()
print(zero_counts)

Danceability          18
Energy                 2
Loudness               2
Speechiness           18
Acousticness           2
Instrumentalness    8985
Liveness               2
Valence               27
Tempo                 18
Duration_min           2
Views                448
Likes                531
Comments             984
Stream               517
EnergyLiveness         2
dtype: int64


In [11]:
#remove rows with zero tempo and duration
df = df[(df['Tempo'] != 0) & (df['Duration_min'] != 0)].copy()

# set flag for zero views and streams
df['zero_engagement_flag'] = (df['Views'] == 0) | (df['Stream'] == 0)
df

,Artist,Track,Album,Album_type,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,Channel,Views,Likes,Comments,Licensed,official_video,Stream,EnergyLiveness,most_playedon,zero_engagement_flag
15166,The Weeknd,Blinding Lights,After Hours,album,0.514,0.730,-5.934,0.0598,0.00146,0.000095,...,TheWeekndVEVO,674164500,8817927,282589,True,True,3386520288,8.138239,Spotify,False
12385,Ed Sheeran,Shape of You,÷ (Deluxe),album,0.825,0.652,-3.183,0.0802,0.58100,0.000000,...,Ed Sheeran,5908398479,31047780,1130327,True,True,3362005201,7.003222,Youtube,False
19082,Lewis Capaldi,Someone You Loved,Divinely Uninspired To A Hellish Extent,album,0.501,0.405,-5.679,0.0319,0.75100,0.000000,...,LewisCapaldiVEVO,586768373,7367091,147565,True,True,2634013335,3.857143,Spotify,False
17847,Post Malone,rockstar (feat. 21 Savage),beerbongs & bentleys,album,0.585,0.520,-6.136,0.0712,0.12400,0.000070,...,PostMaloneVEVO,1060220169,12564657,366520,True,True,2594926619,3.969466,Spotify,False
17848,Post Malone,Sunflower - Spider-Man: Into the Spider-Verse,Hollywood's Bleeding,album,0.755,0.522,-4.368,0.0575,0.53300,0.000000,...,PostMaloneVEVO,1977389041,13749813,331063,True,True,2538329799,7.620438,Spotify,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1585,Scorpions,Always Somewhere,Lovedrive,album,0.511,0.371,-12.074,0.0298,0.07830,0.000864,...,Ricardo Vilarinho,200040,2443,86,False,False,0,3.312500,Youtube,True
13329,Peter Groeger,Kapitel 1.1 - Der Kaiser von Dallas,Der Kaiser von Dallas (Die einzige Wahrheit üb...,album,0.489,0.394,-16.778,0.8700,0.72200,0.000000,...,Jean Bolinder - Topic,119,0,0,True,True,0,1.453875,Youtube,True
11752,Cascada,Everytime We Touch,Everytime We Touch (Premium Edition),album,0.633,0.976,-5.362,0.0519,0.00281,0.000013,...,SteveAATW,181861204,1219373,49314,True,True,0,2.554974,Youtube,True
11754,Cascada,Evacuate The Dancefloor - Radio Edit,Evacuate The Dancefloor,album,0.760,0.696,-5.811,0.0491,0.01950,0.000000,...,Cloud 9 Music,68403117,252132,14739,False,False,0,2.503597,Youtube,True


In [12]:
df.to_csv('outputs/spotify_cleaned.csv', index=False)

# 2. Playlist Extraction

Ports `archive/Playlist Cleaning.ipynb`:

1. **Unique-track whitelist** from the cleaned catalogue (archive cell 0 — originally read `cleandata.csv` and exported `artist_track.csv`; we build it from `spotify_cleaned.csv` produced in §1).
2. **Flatten `playlist.json`** into `(pid, artist_name, track_name)` rows (archive cell 1).
3. **Filter** the flattened extract against the whitelist (archive cell 2 — originally filtered against a separate `unique_tracks.csv`).

The filter here is redundant with the inner-join in §4 (both drop plays whose tracks are not in the catalogue), but it is kept for provenance and produces the same final `taste_profiles.csv` / `user_liked_songs.csv`.

In [13]:
import json
import csv
import pandas as pd

# Step 1 (archive cell 0): build unique (artist, track) whitelist from the cleaned catalogue
catalogue = pd.read_csv('outputs/spotify_cleaned.csv', usecols=['Artist', 'Track'])
unique_tracks = set(zip(catalogue['Artist'].astype(str).str.strip(),
                        catalogue['Track'].astype(str).str.strip()))
print(f'Unique-track whitelist: {len(unique_tracks)} (artist, track) pairs')

# Step 2 (archive cell 1): flatten playlist.json -> (pid, artist_name, track_name)
with open('data/playlist.json') as f:
    data = json.load(f)

raw_rows = [
    {'pid': p['pid'], 'artist_name': t['artist_name'], 'track_name': t['track_name']}
    for p in data['playlists'] for t in p['tracks']
]
playlists_raw = pd.DataFrame(raw_rows)
print(f'Extracted {len(playlists_raw)} track-plays from {playlists_raw["pid"].nunique()} playlists '
      f'(avg {playlists_raw.groupby("pid").size().mean():.2f} songs per playlist)')

# Step 3 (archive cell 2): keep only plays whose (artist, track) is in the whitelist
mask = [(a.strip(), t.strip()) in unique_tracks
        for a, t in zip(playlists_raw['artist_name'], playlists_raw['track_name'])]
playlists_filtered = playlists_raw[mask].reset_index(drop=True)
removed = len(playlists_raw) - len(playlists_filtered)
print(f'Kept {len(playlists_filtered)} rows, removed {removed} '
      f'({removed/len(playlists_raw)*100:.1f}%) not in catalogue')

playlists_filtered.to_csv('outputs/USETHIS_output_filtered.csv', index=False)
print('Saved outputs/USETHIS_output_filtered.csv')

Unique-track whitelist: 19675 (artist, track) pairs
Extracted 67503 track-plays from 1000 playlists (avg 67.50 songs per playlist)
Kept 15011 rows, removed 52492 (77.8%) not in catalogue
Saved outputs/USETHIS_output_filtered.csv
